[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/roberthsu2003/machine_learning/blob/main/AI%E5%8D%94%E4%BD%9C%E6%A9%9F%E5%99%A8%E5%AD%B8%E7%BF%92_%E8%81%B7%E5%A0%B4%E5%AF%A6%E6%88%B0%E7%AF%87/03_%E8%A1%8C%E9%8A%B7_%E5%BB%A3%E5%91%8A%E9%A0%90%E7%AE%97%E6%95%88%E7%9B%8A%E9%A0%90%E6%B8%AC/%E5%BB%A3%E5%91%8A%E9%A0%90%E7%AE%97%E6%95%88%E7%9B%8A%E9%A0%90%E6%B8%AC_%E8%BF%B4%E6%AD%B8%E7%AF%87.ipynb)

# 📈 範例 03：行銷企劃—廣告預算效益與營收預測 (迴歸篇) — AI 伴學實戰筆記本

歡迎來到本實戰筆記本！本教材專為**「品牌行銷主管、數位廣告投手與電商成長黑客（無程式背景）」**設計。
採用**「一段保證可執行的 Python 程式碼 ＋ 一段 AI 伴學互動提問指引」**的雙軌架構：
- 每一段程式碼都經過嚴謹測試，保證直接點擊即可順暢執行。
- 執行完程式碼後，**下方的文字儲存格**會以行銷實務白話解釋程式在做什麼，並提供專屬的 **AI 提問 Prompt（提示詞）**，你可以複製貼給 Copilot、ChatGPT 或 Claude，讓 AI 替你深度解說商業決策邏輯！

---

## 📊 一、行銷資料集介紹 (`ads_sales.csv`)

本數據集記錄了品牌歷年 1,200 個行銷推廣週期的各通路花費與最終營收成效：

| 欄位名稱 | 資料型態 | 數值範例 / 類別 | 行銷業務實務意涵 |
| :--- | :--- | :--- | :--- |
| **Meta臉書廣告_TWD** | 數值型 (`int64`) | 80,000 (元) | 社群曝光與名單型廣告支出 |
| **Google搜尋廣告_TWD** | 數值型 (`int64`) | 100,000 (元) | 高意圖精準導購搜尋廣告 |
| **KOL網紅行銷_TWD** | 數值型 (`int64`) | 50,000 (元) | 部落客與網紅開箱合作費用 |
| **LINE官方推播_TWD** | 數值型 (`int64`) | 15,000 (元) | 會員私域推播訊息成本 |
| **季節檔期** | 類別型 (`object`) | 雙11大促、年中慶、春節連假、平日檔期 | 外部促銷買氣檔期 |
| **品牌搜尋熱度** | 數值型 (`int64`) | 65 (分) | 品牌自然網路聲量指數 |
| **促銷折價券發放數** | 數值型 (`int64`) | 3,000 (張) | 本期折價券派發總量 |
| **預估營收_TWD (目標 y)** | 數值型 (`int64`) | **1,450,000 (元)** | **預測目標：檔期最終營業額（連續數值迴歸）** |

---

## 🧭 二、機器學習全流程地圖

1. **環境配置與繁中字型**：確保圖表顯示中文，避免方塊亂碼 (`□□`)。
2. **資料載入與探索 (EDA)**：檢視廣告花費分佈與營收結構。
3. **特徵相關性與視覺化**：繪製各通路花費與營收的散佈圖與相關係數熱力圖。
4. **特徵工程與 Pipeline 建置**：8:2 切分訓練與測試集，嚴防資料洩漏。
5. **模型選型與 5 折交叉驗證**：比對「線性迴歸」與「隨機森林迴歸器」。
6. **測試集大考與 ROAS 決策**：評估 R²、MAE，輸出模型並模擬新預算案。

---  
### 【步驟 0】：環境準備與繁體中文字型設定

In [ ]:
# [步驟 0 程式碼] 載入套件與設定繁體中文字型
import os, glob
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

try:
    import mplfonts
    from matplotlib import font_manager
    from mplfonts import use_font
    for f in glob.glob(os.path.join(os.path.dirname(mplfonts.__file__), "fonts", "*.[ot]tf")):
        font_manager.fontManager.addfont(f)
    use_font("Noto Sans CJK SC")
except Exception:
    plt.rcParams["font.sans-serif"] = ["Arial Unicode MS", "PingFang TC", "Heiti TC", "Microsoft JhengHei", "sans-serif"]
plt.rcParams["axes.unicode_minus"] = False

print("✅ 套件與中文字型載入成功！準備進入廣告效益迴歸分析。")

#### 💬【步驟 0 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  載入數據分析必備套件，並設定繁體中文字型庫，確保後續繪製營收趨勢圖與預算相關係數圖時軸標籤清晰可讀。

---
💡 **AI 提問小抄：**

1. 📌 **迴歸與分類的本質差異**
   > 「在範例 01 與 02 中我們預測的是『是否成交』與『是否退貨』（分類），而本範例預測的是『營業額金額』（迴歸）。請向非工程師用生活例子說明這兩種問題在機器學習中的根本差別？」


---  
### 【步驟 1】：載入資料與行銷結構探索 (EDA)

In [ ]:
# [步驟 1 程式碼] 讀取 ads_sales.csv 並檢視資料概況
df = pd.read_csv("ads_sales.csv")
print("資料集維度（列, 欄）：", df.shape)
print("\n--- 資料前 5 筆預覽 ---")
display(df.head())
print("\n--- 數值統計摘要（平均值、標準差、最大最小值）---")
display(df.describe().round(0))

#### 💬【步驟 1 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  讀取 `ads_sales.csv`，檢視前 5 筆樣貌，並用 `describe()` 查看廣告預算支出區間與平均營收水準。

---
💡 **AI 提問小抄：**

1. 📌 **數值摘要指標解讀**
   > 「從 `describe()` 輸出的統計摘要中，平均營收約為 136 萬元，但最小只有 20 幾萬、最大超過 300 萬。標準差很大代表什麼商業意涵？為什麼我們需要機器學習而不是直接用平均值來做預算規劃？」


---  
### 【步驟 2】：特徵相關性與通路散佈圖繪製

In [ ]:
# [步驟 2 程式碼] 繪製各廣告預算與營收之散佈圖
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

sns.regplot(data=df, x="Google搜尋廣告_TWD", y="預估營收_TWD", ax=axes[0], color="#1d4ed8", scatter_kws={"alpha": 0.3})
axes[0].set_title("Google搜尋廣告 vs 營收")
axes[0].set_ylabel("營收 (TWD)")

sns.regplot(data=df, x="Meta臉書廣告_TWD", y="預估營收_TWD", ax=axes[1], color="#0f766e", scatter_kws={"alpha": 0.3})
axes[1].set_title("Meta臉書廣告 vs 營收")
axes[1].set_ylabel("")

sns.boxplot(data=df, x="季節檔期", y="預估營收_TWD", ax=axes[2], palette="Set2")
axes[2].set_title("不同季節檔期之營收分佈")
axes[2].set_ylabel("")

plt.tight_layout()
plt.show()

#### 💬【步驟 2 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  繪製 Google 廣告與 Meta 廣告對營收的散佈迴歸趨勢線，並透過箱型圖觀察雙 11 大促與平日檔期的營收差異。

---
💡 **AI 提問小抄：**

1. 📌 **邊際效應與相關性**
   > 「觀察散佈圖上的斜率，為什麼 Google 搜尋廣告的點分佈看起來比 Meta 更緊密地沿著直線上升？這是否代表 Google 廣告對本品牌營收的直接驅動力更強？」


---  
### 【步驟 3】：特徵工程與 Pipeline 建構

In [ ]:
# [步驟 3 程式碼] 切分訓練/測試集，並建立前處理管線
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

X = df.drop(columns=["預估營收_TWD"])
y = df["預估營收_TWD"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

num_cols = ["Meta臉書廣告_TWD", "Google搜尋廣告_TWD", "KOL網紅行銷_TWD", "LINE官方推播_TWD", "品牌搜尋熱度", "促銷折價券發放數"]
cat_cols = ["季節檔期"]

preprocessor = ColumnTransformer(transformers=[
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), cat_cols)
])

print(f"✅ 資料分割完成！訓練集：{X_train.shape[0]} 筆，測試集：{X_test.shape[0]} 筆")

#### 💬【步驟 3 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  將數據集依照 8:2 切分，並將數值預算金額標準化、將檔期類別獨熱編碼，封裝成 `ColumnTransformer`。

---
💡 **AI 提問小抄：**

1. 📌 **數值縮放對迴歸的影響**
   > 「Google 預算通常在幾萬到十幾萬之間，而搜尋熱度只有 20~100。如果不做 `StandardScaler` 標準化，會不會導致模型誤以為『數字大的欄位比較重要』？」


---  
### 【步驟 4】：模型選型與 5 折交叉驗證 (線性迴歸 vs 隨機森林)

In [ ]:
# [步驟 4 程式碼] 使用 5 折交叉驗證評估線性迴歸與隨機森林
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

pipe_lr = Pipeline([("preprocessor", preprocessor), ("regressor", LinearRegression())])
pipe_rf = Pipeline([("preprocessor", preprocessor), ("regressor", RandomForestRegressor(n_estimators=100, max_depth=8, random_state=42))])

scores_lr = cross_val_score(pipe_lr, X_train, y_train, cv=5, scoring="r2")
scores_rf = cross_val_score(pipe_rf, X_train, y_train, cv=5, scoring="r2")

print(f"線性迴歸 5 折交叉驗證 平均 R²: {scores_lr.mean():.4f} (±{scores_lr.std():.4f})")
print(f"隨機森林 5 折交叉驗證 平均 R²: {scores_rf.mean():.4f} (±{scores_rf.std():.4f})")

#### 💬【步驟 4 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  透過 5 折交叉驗證比較線性迴歸與非線性隨機森林的 $R^2$ 解釋力，找出預算綜效預測最佳模型。

---
💡 **AI 提問小抄：**

1. 📌 **線性 vs 非線性的商業實務差異**
   > 「在真實廣告投放中，預算增加通常會有『邊際效應遞減』（例如預算花到一定程度受眾就飽和了）。線性迴歸能捕捉這種飽和現象嗎？為什麼隨機森林更適合？」


---  
### 【步驟 5】：測試集大考與指標解讀 (MAE, RMSE, R²)

In [ ]:
# [步驟 5 程式碼] 在全新測試集評估模型並繪製實際值 vs 預測值對比圖
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

pipe_rf.fit(X_train, y_train)
y_pred = pipe_rf.predict(X_test)

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"=== 測試集最終迴歸評估 ===")
print(f"決定係數 R²: {r2:.4f} (代表模型成功解釋了 {r2*100:.1f}% 的營收變異)")
print(f"平均絕對誤差 MAE: ${mae:,.0f} 元 (平均每筆預測與真實金額的差距)")
print(f"均方根誤差 RMSE: ${rmse:,.0f} 元")

plt.figure(figsize=(7, 6))
plt.scatter(y_test, y_pred, alpha=0.5, color="#2563eb")
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--", lw=2, label="完美預測對角線")
plt.title(f"實際營收 vs 預測營收散佈圖 (R²={r2:.2f})")
plt.xlabel("實際營業額 (TWD)")
plt.ylabel("模型預測營業額 (TWD)")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()

#### 💬【步驟 5 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  計算測試集的 $R^2$、MAE 與 RMSE，並繪製對角線散佈圖。點越貼近紅色虛線，代表預估金額越精準。

---
💡 **AI 提問小抄：**

1. 📌 **指標實務應用**
   > 「如果行銷長問我：『這套預估模型到底準不準？』，我該如何用 MAE 告訴他：『我們預估一百多萬的營收，平均誤差只有幾萬元』？」


---  
### 【步驟 6】：新預算方案即時預估 (Inference) 與 ROAS 分析

In [ ]:
# [步驟 6 程式碼] 保存模型並模擬新行銷檔期預算效益預測
joblib.dump(pipe_rf, "model.joblib")
print("✅ 模型管線已成功保存至 model.joblib！")

# 模擬規劃下個月「雙11大促」的廣告預算投放案
plan = pd.DataFrame([{
    "Meta臉書廣告_TWD": 120000,
    "Google搜尋廣告_TWD": 150000,
    "KOL網紅行銷_TWD": 100000,
    "LINE官方推播_TWD": 25000,
    "季節檔期": "雙11大促",
    "品牌搜尋熱度": 85,
    "促銷折價券發放數": 8000
}])

loaded_model = joblib.load("model.joblib")
pred_sales = loaded_model.predict(plan)[0]
total_budget = 120000 + 150000 + 100000 + 25000
est_roas = pred_sales / total_budget

print(f"\n📢 下月行銷案試算報告：")
print(f"總廣告預算投入：${total_budget:,} 元")
print(f"預估創造營業額：${pred_sales:,.0f} 元")
print(f"預估廣告投資報酬率 (ROAS)：{est_roas:.2f} 倍")

#### 💬【步驟 6 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  將模型封裝儲存，並即時試算雙 11 大促方案的預估營收與預期 ROAS 投報率。

---
💡 **AI 提問小抄：**

1. 📌 **行銷決策演繹**
   > 「如果我想透過 Python 寫一個自動化最佳化腳本，在總預算固定為 30 萬元的限制下，自動搜尋出能讓預估營收最大化的 Meta、Google、KOL 配比，這在資料科學上該怎麼做？」
